In [1]:
import os

os.environ["KERAS_HOME"] = r"D:\keras_cache"
os.environ["TFHUB_CACHE_DIR"] = r"D:\keras_cache"

os.makedirs(r"D:\keras_cache\models", exist_ok=True)

print("Keras cache moved to D drive")

Keras cache moved to D drive


In [2]:

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import json

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping


In [3]:

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

X = np.concatenate((X_train, X_test))
y = np.concatenate((y_train, y_test))

NUM_CLASSES = 10

print("Total images:", len(X))
print("Number of categories:", NUM_CLASSES)
print("Image shape:", X[0].shape)


Total images: 70000
Number of categories: 10
Image shape: (28, 28)


In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


Training: (47600, 28, 28)
Validation: (8400, 28, 28)
Testing: (14000, 28, 28)


In [5]:

IMG_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 5

def load_image(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.expand_dims(image, axis=-1)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.image.grayscale_to_rgb(image)
    return image, label


def make_dataset(images, labels, preprocess=True):

    dataset = tf.data.Dataset.from_tensor_slices(
        (images, labels)
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if preprocess:
        dataset = dataset.map(
            lambda x, y: (preprocess_input(x), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(X_train, y_train)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

print("Image pipeline created successfully!")


Image pipeline created successfully!


In [6]:

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

print("VGG16 loaded successfully!")


VGG16 loaded successfully!


In [7]:

base_model.trainable = False

print("Pretrained VGG16 layers frozen!")


Pretrained VGG16 layers frozen!


In [8]:

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)

x = Dropout(0.4)(x)

output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 128, 128, 3)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_conv1 (Conv2D)         │ (None, 128, 128, 64)  │        1,792 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_conv2 (Conv2D)         │ (None, 128, 128, 64)  │       36,928 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_pool (MaxPooling2D)    │ (None, 64, 64, 64)    │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv1 (Conv2D)         │ (None, 64, 64, 128)   │       73,856 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv2 (Conv2D)         │ (None, 64, 64, 128)   │      147,584 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_pool (MaxPooling2D)    │ (None, 32, 32, 128)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv1 (Conv2D)         │ (None, 32, 32, 256)   │      295,168 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv2 (Conv2D)         │ (None, 32, 32, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv3 (Conv2D)         │ (None, 32, 32, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_pool (MaxPooling2D)    │ (None, 16, 16, 256)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv1 (Conv2D)         │ (None, 16, 16, 512)   │    1,180,160 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv2 (Conv2D)         │ (None, 16, 16, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv3 (Conv2D)         │ (None, 16, 16, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_pool (MaxPooling2D)    │ (None, 8, 8, 512)     │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv1 (Conv2D)         │ (None, 8, 8, 512)     │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv2 (Conv2D)         │ (None, 8, 8, 512)     │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv3 (Conv2D)         │ (None, 8, 8, 512)     │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_pool (MaxPooling2D)    │ (None, 4, 4, 512)     │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ global_average_pooling2d      │ (None, 512)           │            0 │
│ (GlobalAveragePooling2D)      │                       │              │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense (Dense)                 │ (None, 128)           │       65,664 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dropout (Dropout)             │ (None, 128)           │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_1 (Dense)               │ (None, 10)            │        1,290 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 14,781,642 (56.39 MB)

 Trainable params: 66,954 (261.54 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [9]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop]
)

Epoch 1/5


D:\retailmind_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


2975/2975 ━━━━━━━━━━━━━━━━━━━━ 1802s 605ms/step - accuracy: 0.8008 - loss: 0.6125 - val_accuracy: 0.8723 - val_loss: 0.3456
Epoch 2/5
2975/2975 ━━━━━━━━━━━━━━━━━━━━ 1586s 533ms/step - accuracy: 0.8534 - loss: 0.4097 - val_accuracy: 0.8830 - val_loss: 0.3079
Epoch 3/5
2975/2975 ━━━━━━━━━━━━━━━━━━━━ 1562s 525ms/step - accuracy: 0.8659 - loss: 0.3743 - val_accuracy: 0.8911 - val_loss: 0.3075
Epoch 4/5
2975/2975 ━━━━━━━━━━━━━━━━━━━━ 1826s 614ms/step - accuracy: 0.8717 - loss: 0.3524 - val_accuracy: 0.8888 - val_loss: 0.3074
Epoch 5/5
2975/2975 ━━━━━━━━━━━━━━━━━━━━ 1896s 618ms/step - accuracy: 0.8788 - loss: 0.3323 - val_accuracy: 0.8913 - val_loss: 0.2964


In [11]:
# Unfreeze VGG16
base_model.trainable = True

# Freeze all layers except the last 4
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Recompile with a very low learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Fine-tune the model
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=[early_stop]
)

Epoch 1/3
2975/2975 ━━━━━━━━━━━━━━━━━━━━ 2709s 910ms/step - accuracy: 0.9083 - loss: 0.2540 - val_accuracy: 0.9207 - val_loss: 0.2286
Epoch 2/3
2975/2975 ━━━━━━━━━━━━━━━━━━━━ 2628s 878ms/step - accuracy: 0.9277 - loss: 0.1964 - val_accuracy: 0.9265 - val_loss: 0.2185
Epoch 3/3
2975/2975 ━━━━━━━━━━━━━━━━━━━━ 2808s 927ms/step - accuracy: 0.9411 - loss: 0.1560 - val_accuracy: 0.9299 - val_loss: 0.2326


In [12]:
vgg_loss, vgg_accuracy = model.evaluate(test_ds)

print("VGG16 Test Loss:", vgg_loss)
print("VGG16 Test Accuracy:", vgg_accuracy)

875/875 ━━━━━━━━━━━━━━━━━━━━ 554s 634ms/step - accuracy: 0.9217 - loss: 0.2318
VGG16 Test Loss: 0.23177824914455414
VGG16 Test Accuracy: 0.9217143058776855


In [3]:
print(tf.config.list_physical_devices('GPU'))

[]


In [4]:
print("TensorFlow version:", tf.__version__)
print("Available devices:", tf.config.list_physical_devices())
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow version: 2.21.0
Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
GPU available: False


In [5]:
print(tf.config.list_physical_devices('GPU'))

[]


In [6]:
print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
vgg_loss, vgg_accuracy = model.evaluate(test_ds)

print("VGG16 Test Loss:", vgg_loss)
print("VGG16 Test Accuracy:", vgg_accuracy)

In [ ]:
day6_model = load_model("../Models/fashion_cnn.keras")

day6_test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

day6_test_ds = day6_test_ds.map(
    lambda image, label: (
        tf.cast(tf.expand_dims(image, -1), tf.float32) / 255.0,
        label
    )
).batch(32).prefetch(tf.data.AUTOTUNE)

day6_loss, day6_accuracy = day6_model.evaluate(day6_test_ds)

print("Day 6 CNN Accuracy:", day6_accuracy)
print("Day 7 VGG16 Accuracy:", vgg_accuracy)

In [9]:
MODELS_DIR = Path("../Models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model.save(MODELS_DIR / "fashion_vgg16.keras")

print("Day 7 VGG16 model saved successfully!")

NameError: name 'model' is not defined